# Administrator one-time setup: Cantus model weights

## 管理者一次性準備：Cantus 模型權重

Run this notebook **once** before any downstream user opens `notebooks/task_template.ipynb`. The administrator mirrors `google/gemma-4-E2B-it` and `google/gemma-4-E4B-it` from Hugging Face Hub into a Drive directory you control. Downstream users then mount the same Drive and load the weights through `mount_drive_and_load(variant="E4B")` — they never touch Hugging Face Hub, so they avoid anonymous rate limits and corporate / campus firewall issues.

> **Always download both variants.** The default for downstream use is **E4B** (4B parameters, ~2 GB VRAM after 4-bit quantisation, comfortable on a Colab T4) because 4B is significantly more compliant under grammar-constrained tool calling than 2B. **E2B is experimental — see `docs/cookbook/errors.md`** (relative to this cantus repo). E2B remains a legal choice for very-low-VRAM scenarios or for demonstrating small-model limits, but Cantus retries automatically when E2B short-circuits an empty `final_answer`. Cantus v0.1.2+ ships schema-level + runtime-level non-empty answer enforcement, so the EventStream may legitimately contain `ValidationErrorObservation(validator_name="non_empty_final_answer", ...)` entries during retry — this is by design, not a bug.

> The Cantus framework lives at `https://github.com/schola-cantorum/cantus`. Downstream users run `pip install git+https://github.com/schola-cantorum/cantus@v0.1.3` from inside `notebooks/task_template.ipynb`. **This admin notebook does NOT install Cantus itself** — its only job is to put Gemma 4 weights on Drive.

## No HF_TOKEN required

**Gemma 4 is Apache 2.0 and ungated on Hugging Face.** Anyone can `snapshot_download` anonymously — no HF account, no licence acceptance, no `HF_TOKEN`. This notebook downloads anonymously by default.

Trade-off: anonymous requests face stricter rate limits. If you hit HTTP 429 mid-download, add an `HF_TOKEN` to Colab Secrets (any free read token suffices) to raise the limit. **This step is optional.**

## Before running this notebook

Decide where on Drive you want the weights:

- **MyDrive** (personal / single-user setup) — e.g. `/content/drive/MyDrive/cantus_models`.
- **Shared Drive** (team / classroom / lab setup) — create a Workspace Shared Drive, add the downstream users / group with Viewer permission, and point `SHARED_DRIVE_PATH` below at `Shareddrives/<your-shared-drive-name>/models/`.

## Expected download size

| Variant | safetensors size | Download time (Colab → HF CDN) | Role |
| --- | --- | --- | --- |
| E4B-it | ~16 GB | 4-8 minutes | **Default** (recommended) |
| E2B-it | ~10 GB | 2-5 minutes | experimental (small-model demos) |

Total ~26 GB; ensure your Drive has the capacity. **Download both variants** so downstream users can switch `model_variant` without hitting `ModelNotFoundError`.

## Why the directory name ends in `-4bit` even though we download raw weights

The directory name is a Cantus convention (`cantus.model.loader.VARIANT_TO_DIRNAME`). It signals that the weights at that path will be loaded in 4-bit mode. The actual files on disk are Google's raw safetensors (bf16 / fp16); **`bitsandbytes` performs the 4-bit quantisation dynamically at downstream load time**. This keeps `cantus.model.loader` simple and lets downstream users switch to 8-bit or fp16 by overriding the loader if they want.

If you would rather store pre-quantised weights on Drive (smaller, but requires loader changes), see the last cell labelled "Advanced: pre-quantised storage".

## Step 1: mount Drive and set the target path

In [ ]:
from google.colab import drive, userdata
import os
from pathlib import Path

drive.mount('/content/drive', force_remount=False)

#@title Set Drive path { display-mode: "form" }
SHARED_DRIVE_PATH = "/content/drive/MyDrive/cantus_models" #@param {type:"string"}

Path(SHARED_DRIVE_PATH).mkdir(parents=True, exist_ok=True)
print(f"Target path: {SHARED_DRIVE_PATH}")
print(f"Path exists: {Path(SHARED_DRIVE_PATH).exists()}")

## Step 2: Hugging Face login (optional)

Gemma 4 is ungated — downloading **does not require** login. This cell skips login by default; only when an `HF_TOKEN` Colab Secret is already set does it opportunistically log in to raise the anonymous rate limit. Without `HF_TOKEN` everything still works; proceed to Step 3.

In [ ]:
!pip install -q -U huggingface_hub

# Gemma 4 is ungated (Apache 2.0); downloading does not require a token.
# Only opportunistically log in when HF_TOKEN is present, to avoid anonymous rate limits.
from huggingface_hub import login

try:
    hf_token = userdata.get('HF_TOKEN')
    if hf_token:
        login(token=hf_token, add_to_git_credential=False)
        print("✓ HF_TOKEN found in Colab Secrets — logged in (raises anonymous rate limit)")
    else:
        print("ℹ HF_TOKEN secret is empty — downloading anonymously (Gemma 4 does not need a token).")
except (userdata.SecretNotFoundError, userdata.NotebookAccessError):
    print("ℹ HF_TOKEN not set — downloading anonymously (Gemma 4 does not need a token).")
    print("  If you hit HTTP 429 mid-download, add HF_TOKEN via the left-side 🔑 panel and re-run.")

## Step 3: download E2B and E4B

`snapshot_download` mirrors the entire HF repo into Drive. `hf_transfer` lifts download speed from ~30 MB/s to ~200 MB/s.

**First run takes a few minutes.** Safe to interrupt — re-running resumes (already-downloaded files are skipped).

In [ ]:
# Enable hf_transfer for faster downloads
!pip install -q hf_transfer
import os
os.environ['HF_HUB_ENABLE_HF_TRANSFER'] = '1'

from huggingface_hub import snapshot_download

# Cantus directory naming convention (cantus.model.loader.VARIANT_TO_DIRNAME)
VARIANTS = [
    ("E2B", "google/gemma-4-E2B-it", "gemma-4-E2B-it-4bit"),
    ("E4B", "google/gemma-4-E4B-it", "gemma-4-E4B-it-4bit"),
]

for variant, repo_id, dirname in VARIANTS:
    target = f"{SHARED_DRIVE_PATH}/{dirname}"
    print(f"\n=== Downloading {variant}: {repo_id} → {target} ===")
    snapshot_download(
        repo_id=repo_id,
        local_dir=target,
        local_dir_use_symlinks=False,  # Drive does not support symlinks
        # Skip PyTorch / Flax / TF / msgpack formats (keep safetensors + tokenizer + config only)
        ignore_patterns=["*.bin", "*.pt", "*.msgpack", "*.h5", "*.gguf"],
        max_workers=4,
    )
    print(f"✓ {variant} downloaded")

## Step 4: verify the downloads

Check that each variant directory contains the required files.

In [ ]:
REQUIRED_FILES = [
    "config.json",
    "tokenizer_config.json",
]

all_ok = True
for variant, _repo, dirname in VARIANTS:
    target = Path(f"{SHARED_DRIVE_PATH}/{dirname}")
    print(f"\n=== {variant}: {target} ===")
    for fname in REQUIRED_FILES:
        path = target / fname
        ok = path.exists()
        all_ok = all_ok and ok
        print(f"  {'✓' if ok else '✗'} {fname}")
    safetensors = list(target.glob("*.safetensors"))
    if safetensors:
        total_mb = sum(p.stat().st_size for p in safetensors) / 1024 / 1024
        print(f"  ✓ {len(safetensors)} safetensors files ({total_mb:.0f} MB total)")
    else:
        all_ok = False
        print("  ✗ No safetensors files!")

if all_ok:
    print("\n🎉 All downloads complete. You can now share this Drive directory with downstream users (Viewer permission).")
else:
    print("\n❌ Missing files — re-run Step 3.")

## Step 5: smoke test (optional)

Load the model from the Drive path once to confirm the files are intact and 4-bit quantisation works end-to-end. **This step requires a GPU runtime** (~30 seconds).

In [ ]:
!pip install -q -U transformers bitsandbytes accelerate

import torch
from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig

test_variant = "E2B"  # Use the smaller variant to save time
test_path = f"{SHARED_DRIVE_PATH}/gemma-4-{test_variant}-it-4bit"

print(f"Loading from {test_path}...")
quant = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_compute_dtype=torch.bfloat16,
    bnb_4bit_use_double_quant=True,
)
tokenizer = AutoTokenizer.from_pretrained(test_path)
model = AutoModelForCausalLM.from_pretrained(
    test_path,
    quantization_config=quant,
    device_map="auto",
)

vram_gb = torch.cuda.memory_allocated() / 1024**3
print(f"✓ Loaded; VRAM usage {vram_gb:.2f} GB")

# Generate a short response.
# Note: apply_chat_template for multimodal tokenizers (Gemma 4 / Gemma 3n) returns a BatchEncoding,
# not a Tensor. Use return_dict=True + **inputs unpacking — same canonical pattern as cantus.model.loader.
msgs = [{"role": "user", "content": "Describe photosynthesis in one sentence."}]
inputs = tokenizer.apply_chat_template(
    msgs,
    return_tensors="pt",
    add_generation_prompt=True,
    return_dict=True,
).to(model.device)
out = model.generate(**inputs, max_new_tokens=80, do_sample=False)
response = tokenizer.decode(out[0][inputs["input_ids"].shape[1]:], skip_special_tokens=True)
print("\nModel response:")
print(response)

## Advanced: pre-quantised storage (save Drive space)

The default flow stores raw weights on Drive (~26 GB across both variants) and lets `bitsandbytes` quantise at load time. If you want pre-quantised weights on Drive (~10 GB across both variants) you can do:

```python
# Load raw weights + 4-bit
model = AutoModelForCausalLM.from_pretrained(
    test_path,
    quantization_config=BitsAndBytesConfig(load_in_4bit=True, bnb_4bit_compute_dtype=torch.bfloat16),
    device_map="auto",
)
# save_pretrained writes the quantised state back
model.save_pretrained(f"{SHARED_DRIVE_PATH}/gemma-4-{test_variant}-it-4bit-quant")
```

**Caveats**: requires transformers `save_pretrained` quantised-state support (≥ 4.41). After writing pre-quantised weights, downstream loaders must omit `quantization_config` (otherwise double-quantisation errors). This requires editing `cantus.model.loader` to detect a `quant_config.json` marker and skip re-quantisation.

**Cantus default does not adopt** this path because it complicates the loader. Switch only when Drive capacity is a real constraint.